 # 0.04 — NULL Values by Year

 Breaks down NULL rates per feature column for each training year (2019, 2021, 2026).
 Shows which NULLs are era-specific data gaps vs structural issues that persist
 across all years — informs which years to include in the training window.

 Authored as a `# %%` .py file (clean git diffs). Export to `.ipynb` with outputs via:
   Command Palette → "Jupyter: Export Current Python File as Jupyter Notebook"

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import psycopg2

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
from citibike.config import DB_CONFIG  # noqa: E402
from model_training.build_training_features_pandas import (  # noqa: E402
    INSERT_COLUMNS, INT64_COLUMNS, BOOL_COLUMNS,
)

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 20)

In [ ]:
TABLE = "training_features"
YEARS = [2019, 2021, 2026]

PK      = ["station_id", "timestamp", "horizon_minutes"]
TARGETS = ["bikes_available_at_horizon", "bike_available_binary"]
TEXT_COLS = ["season", "station_role"]

feature_cols = [c for c in INSERT_COLUMNS if c not in PK and c not in TARGETS]

def q(col):
    return f'"{col}"' if col == "timestamp" else col

conn = psycopg2.connect(**DB_CONFIG)
cur  = conn.cursor()

# Row count per year (single query)
cur.execute(f"""
    SELECT EXTRACT(YEAR FROM "timestamp")::int AS yr, COUNT(*) AS total
    FROM {TABLE}
    WHERE EXTRACT(YEAR FROM "timestamp") = ANY(%(years)s)
    GROUP BY yr ORDER BY yr;
""", {"years": YEARS})
year_totals = {yr: total for yr, total in cur.fetchall()}
print("Row counts per year:")
for yr, total in year_totals.items():
    print(f"  {yr}: {total:,}")

Row counts per year:
  2019: 35,918,098
  2021: 36,203,726
  2026: 13,794,287


 ## NULL rates per feature, per year

 One query per year — each is a single full scan of that year's rows.
 Columns sorted by their 2019 NULL rate (highest first) so the worst offenders
 appear at the top.

In [ ]:
null_exprs = ", ".join(
    f"SUM(CASE WHEN {q(c)} IS NULL THEN 1 ELSE 0 END) AS col_{i}"
    for i, c in enumerate(feature_cols)
)

results = {}
for yr in YEARS:
    cur.execute(f"""
        SELECT {null_exprs}
        FROM {TABLE}
        WHERE EXTRACT(YEAR FROM "timestamp") = %(yr)s;
    """, {"yr": yr})
    row = cur.fetchone()
    total = year_totals.get(yr, 1)
    results[yr] = {
        c: round(100.0 * (row[i] or 0) / total, 1)
        for i, c in enumerate(feature_cols)
    }

null_pct = pd.DataFrame(results, columns=YEARS)
null_pct.index = feature_cols

# Sort by 2019 NULL % descending, drop rows that are 0% across all years
null_pct = null_pct[null_pct.max(axis=1) > 0].sort_values(2019, ascending=False)
null_pct.columns = [f"{yr} (%)" for yr in YEARS]
null_pct

,2019 (%),2021 (%),2026 (%)
rate_of_change_30min,100.0,100.0,100.0
is_within_400m,100.0,100.0,0.0
entrance_count_800m,100.0,100.0,0.0
entrance_count_400m,100.0,100.0,0.0
rate_of_change_20min,100.0,100.0,100.0
rate_of_change_10min,100.0,100.0,100.0
nearest_entrance_dist_m,100.0,100.0,15.6
change_classic_12hr,76.4,24.4,4.3
change_ebikes_12hr,76.4,24.4,4.3
change_ebikes_6hr,76.2,23.1,2.4


 ## Summary: which NULLs are era-specific vs structural?

 - **Same across all years** → structural (data never existed; encode-as-missing)
 - **High in 2019, low in 2026** → legacy era gap (no fix without external data)
 - **High in 2019, gone in 2021/2026** → year-specific data coverage issue

In [ ]:
df = null_pct.copy()
df.columns = [2019, 2021, 2026]

era_specific  = df[(df[2019] > 50) & (df[2026] < 20)].index.tolist()
structural    = df[(df[2019] > 50) & (df[2026] > 50)].index.tolist()
low_all_years = df[df.max(axis=1) < 10].index.tolist()

print("ERA-SPECIFIC NULLs (high in 2019, populated in 2026):")
print("  → caused by data unavailability in the pre-2021 era")
for c in era_specific:
    print(f"  {c:<40} 2019={df.loc[c,2019]:.1f}%  2026={df.loc[c,2026]:.1f}%")

print("\nSTRUCTURAL NULLs (high across all years):")
print("  → data genuinely doesn't exist; encode-as-missing, do not impute")
for c in structural:
    print(f"  {c:<40} 2019={df.loc[c,2019]:.1f}%  2026={df.loc[c,2026]:.1f}%")

print("\nLOW NULLs across all years (< 10%):")
print("  → well-populated; minor imputation only for linear models")
for c in low_all_years:
    print(f"  {c:<40} 2019={df.loc[c,2019]:.1f}%  2021={df.loc[c,2021]:.1f}%  2026={df.loc[c,2026]:.1f}%")

ERA-SPECIFIC NULLs (high in 2019, populated in 2026):
  → caused by data unavailability in the pre-2021 era
  is_within_400m                           2019=100.0%  2026=0.0%
  entrance_count_800m                      2019=100.0%  2026=0.0%
  entrance_count_400m                      2019=100.0%  2026=0.0%
  nearest_entrance_dist_m                  2019=100.0%  2026=15.6%
  change_classic_12hr                      2019=76.4%  2026=4.3%
  change_ebikes_12hr                       2019=76.4%  2026=4.3%
  change_ebikes_6hr                        2019=76.2%  2026=2.4%
  change_classic_6hr                       2019=76.2%  2026=2.4%
  change_ebikes_3hr                        2019=74.9%  2026=1.2%
  change_classic_3hr                       2019=74.9%  2026=1.2%
  change_classic_1hr                       2019=73.7%  2026=0.5%
  change_ebikes_1hr                        2019=73.7%  2026=0.5%
  num_bikes_disabled                       2019=69.4%  2026=0.0%
  num_ebikes_available                    

In [ ]:
conn.close()
print("Done.")

Done.
